In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
from langchain_ollama import ChatOllama

from CIT.evaluation.utils import load_jsonl

In [ ]:
JUDGE_PROMPT = """
You are a judge that rates answers. You will be given a user question, a context, a reference answer and a generated answer.
Your task is to provide a 'total rating' scoring how well the generated answer answers the user question regarding the context and the true answer.
Give your answer as an integer on a scale of 1 to 3.
Remember that the information in the generated answer must be in the context.

Here is the scale you should use to build your answer:
1: The generated answer is terrible: it is false, or it contains information that is not in the context, like hallucinations
2: The generated answer contains some true information but it does not fully answer the question, knowing the reference answer answers the question
3: The generated answer is helpful: it contains true information and answers the question, it is complete
If the reference answer says that it does not have the information to answer the question, then the generated answer must also mention that it does not have the information to answer the question, otherwise it will be rated as 1.
If both the reference answer and the generated answer say that they do not have the information to answer the question, then you must rate it as 3.


Now here are the question, the context the reference answer and the generated answer.

User Question::: {question}
Context::: <begin of context> {context}</end of context>
Reference answer::: {true_answer}
Generated answer::: {generated_answer}

Provide your feedback as follows, this is very important. The format of your answer must absolutely follow the following template. Do not add anything else after your rating:
Template:
Feedback:::
Evaluation: your short rationale for the rating, as a text
Total rating: the integer between 1 and 3 (x/3)


Do not add anything else than the feedback and the total rating.
Remember that your goal is to provide a rating between 1 and 3 for the generated answer compared to the reference answer, considering the context.
It is not to solve the question or to provide a complete answer.
Do not try to solve the question. You are a judge.
Feedback:::
The rating between 1 and 3.
Total rating: """



In [ ]:

def evaluate_sample(sample, llm_judge):
    """
    Evaluate a single generated answer by comparing it to the ground truth using a judging LLM.

    Args:
        sample (dict): A dictionary containing:
            - "question": The input question.
            - "RAG_answer": The answer generated by the RAG system.
            - "context": The source context used for generation.
            - "answer": The ground-truth reference answer.
        llm_judge: An LLM instance (e.g. via Ollama) used to evaluate the quality of the generated answer.
        few_shots (bool, optional): Whether to use few-shot prompting with examples. Defaults to False.

    Returns:
        sample (dict): The updated sample dictionary with an added rating score:
            - "rating" if few_shots is False.
            - "rating_few_shots" if few_shots is True.
            If the generated answer is invalid ("Error") or parsing fails, the rating is set to NaN.
    """

    question = sample["question"]
    generated_answer = sample["RAG_answer"]
    if "context" in sample:
        context = sample["context"]
    elif "real_retrieved_context" in sample:
        context = sample["real_retrieved_context"]
        print("Using real_retrieved_context")
    elif "retrieved_context" in sample:
        context = sample["retrieved_context"]
    else:
        context = "No context provided"
        
    if generated_answer == "Error":
        sample["rating"] = np.nan
        return sample
    true_answer = sample["answer"]
    prompt = JUDGE_PROMPT.format(
        question=question,
        true_answer=true_answer,
        generated_answer=generated_answer,
        context=context,

        )

    feedback = llm_judge.invoke(prompt)  # call the judge llm
    print(feedback.content)  # print the feedback for debugging
    try:  # parse rating
        pattern = r"Total rating: ?\(?(\d)(?:/5)?\)?"
        match = re.search(pattern, feedback.content)
        if match:
            rating = int(match.group(1))
        else:
            print(f"prompt: {prompt}")
            print(f"feedback: {feedback.content}")
            rating = np.nan
    except Exception as e:
        print(e)
        print("New eroooooooooooooooooooor")

    sample["rating"] = rating
    return sample

In [ ]:
llm_judge=ChatOllama(model="llama3.1:8b",temperature=0, max_tokens=50)

In [ ]:
answers_path="./src/CIT/training/models/cv/data/test_answers/ft_02.06_with_at_least_1_url_eval_all_data/ft/answers_fold_2.jsonl"
samples = load_jsonl(answers_path)

In [ ]:
sample=np.random.choice(samples)
print(f"Sample question: {sample['question']}")
print(f"Sample generated answer: {sample['RAG_answer']}")
print(f"Sample true answer: {sample['answer']}")
sample = evaluate_sample(sample, llm_judge)
print(f"Sample rating: {sample['rating']}")


In [ ]:
llm_70b=ChatOllama(model="llama3.3:70b",temperature=0, max_tokens=3)

In [ ]:
750*0.5/60

In [ ]:
#sample=np.random.choice(samples)
print(f"Sample question: {sample['question']}")
print(f"Sample generated answer: {sample['RAG_answer']}")
print(f"Sample true answer: {sample['answer']}")
sample = evaluate_sample(sample, llm_70b)
print(f"Sample rating: {sample['rating']}")

# Check answers with wrong format

In [ ]:
import json

In [ ]:
path_wrong_sample="./src/CIT/evaluation/results/llm_judge_examples/wrong_cases/example_d04a3cb6-8b9a-43d3-95c3-09b5da7891ea"

with open(path_wrong_sample, "r") as f:
    wrong_sample = json.load(f)

In [ ]:
for key in wrong_sample:
    print(key)

In [ ]:
sample=wrong_sample
print(f"Sample question: {sample['question']}")
print(f"Sample generated answer: {sample['RAG_answer']}")
print(f"Sample true answer: {sample['answer']}")
print("---"* 20)
sample = evaluate_sample(sample, llm_judge)
print(f"Sample rating: {sample['rating']}")

# Check correlation with recall and precision, and quality

In [ ]:
import pandas as pd

## 1-3 scale metric

In [ ]:
path_results="./src/CIT/evaluation/results/llm_judge_examples/ft_02.06_with_at_least_1_url_eval_all_data/ft/answers_fold_1.jsonl"
results = load_jsonl(path_results)
df = pd.DataFrame(results,columns=["rating","precision","recall","quality"])

In [ ]:
#bar plot with with quality and rating -1 counts
plt.figure(figsize=(10, 6))
plt.bar((df["quality"]+1).value_counts().index, (df["quality"]+1).value_counts().values, alpha=0.5, label="Our metric")
plt.bar(df["rating"].value_counts().index, df["rating"].value_counts().values, alpha=0.5, label="Rating LLM judge)")
#legend
plt.xticks(np.arange(1, 4), labels=["1", "2", "3"])
plt.xlabel("Rating")
plt.ylabel("Count")
plt.title("LLM Judge Ratings vs Our Metric Ratings")
plt.legend(["Our metric", "Rating LLM judge"])
plt.show()

## with 0-1 scale

In [ ]:
import pandas as pd

from CIT.evaluation.utils import load_jsonl

In [ ]:
path_results="./src/CIT/evaluation/results/llm_judge_examples/ft_02.06_with_at_least_1_url_eval_all_data/ft/answers_fold_1_binary.jsonl"
results = load_jsonl(path_results)
df = pd.DataFrame(results,columns=["rating","precision","recall","quality","retrieved_context","RAG_answer","question","answer"])
len(df)

In [ ]:
sample=np.random.choice(results)
print(f"Sample question: {sample['question']}")
print(f"Sample generated answer: {sample['RAG_answer']}")
print(f"Sample true answer: {sample['answer']}")
print("---"* 20)
print(f"Sample quality: {sample['quality']}")
print(f"Sample rating: {sample['rating']}")

In [ ]:
df["quality_binary"] = df["quality"].apply(lambda x: 1 if x >= 1 else 0)

In [ ]:
df[["rating","precision","recall","quality","quality_binary"]].corr().style.background_gradient(cmap='coolwarm', axis=0).format(precision=2)

In [ ]:
#confusion matrix with rating and quality_binary
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(df["quality_binary"], df["rating"])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0,1], yticklabels=[0, 1])
plt.xlabel('Rating')
plt.ylabel('Quality Binary')
plt.title('Confusion Matrix: Quality Binary vs Rating')
plt.show()

In [ ]:
#bar plot with with quality and rating -1 counts
plt.figure(figsize=(10, 6))
plt.bar((df["quality_binary"]).value_counts().index, (df["quality_binary"]).value_counts().values, alpha=0.5, label="Our metric")
plt.bar(df["rating"].value_counts().index, df["rating"].value_counts().values, alpha=0.5, label="Rating LLM judge)")
#legend
plt.xticks(np.arange(0, 2), labels=["0", "1"])
plt.xlabel("Rating")
plt.ylabel("Count")
plt.title("LLM Judge Ratings vs Our Metric Ratings")
plt.legend(["Our metric", "Rating LLM judge"])
plt.show()

## see if similar for all_folds

In [ ]:
import os
path_folder="./src/CIT/evaluation/results/llm_judge_examples/ft_02.06_with_at_least_1_url_eval_all_data/ft/"

files = os.listdir(path_folder)
files = [f for f in files if f.endswith("binary.jsonl")]
results = []
for file in files:
    path_results=os.path.join(path_folder, file)
    results_fold = load_jsonl(path_results)
    for sample in results_fold:
        sample["fold"] = int(file.split("_")[-2])
    results.extend(results_fold)

df = pd.DataFrame(results,columns=["rating","precision","recall","quality","fold"])
df["quality_binary"] = df["quality"].apply(lambda x: 1 if x >= 1 else 0)
num_folds= df["fold"].nunique()
#plot the distribution of quality_binary and rating for each fold
fig,ax=plt.subplots(num_folds//2+1, 2, figsize=(15, 10), sharex=True, sharey=False)
for i, fold in enumerate(df["fold"].unique()):
    df_fold = df[df["fold"] == fold]
    ax[i//2, i%2].bar((df_fold["quality_binary"]).value_counts().index, (df_fold["quality_binary"]).value_counts().values, alpha=0.5, label="Our metric")
    ax[i//2, i%2].bar(df_fold["rating"].value_counts().index, df_fold["rating"].value_counts().values, alpha=0.5, label="Rating LLM judge)")
    ax[i//2, i%2].set_title(f"Fold {fold}")
    ax[i//2, i%2].set_xticks(np.arange(0, 2), labels=["0", "1"])
    ax[i//2, i%2].set_xlabel("Rating")
    ax[i//2, i%2].set_ylabel("Count")
    ax[i//2, i%2].legend(["Our metric", "Rating LLM judge"])
fig.delaxes(ax[-1, -1])  # Remove the last empty subplot if num_folds is odd
plt.tight_layout()
plt.show()

In [ ]:
import os
path_folder="./src/CIT/evaluation/results/llm_judge_examples/ft_02.06_with_at_least_1_url_eval_all_data/ft/"

files = os.listdir(path_folder)
files = [f for f in files if f.endswith("binary.jsonl")]
results = []
for file in files:
    path_results=os.path.join(path_folder, file)
    results_fold = load_jsonl(path_results)
    for sample in results_fold:
        sample["fold"] = int(file.split("_")[-2])
    results.extend(results_fold)

df = pd.DataFrame(results,columns=["rating","precision","recall","quality","fold"])
df["quality_binary"] = df["quality"].apply(lambda x: 1 if x >= 1 else 0)
num_folds= df["fold"].nunique()
#plot the distribution of quality_binary and rating for each fold
fig,ax=plt.subplots(num_folds//2+1, 2, figsize=(15, 10), sharex=True, sharey=False)
for i, fold in enumerate(df["fold"].unique()):
    df_fold = df[df["fold"] == fold]
    ax[i//2, i%2].bar((df_fold["quality_binary"]).value_counts().index, (df_fold["quality_binary"]).value_counts().values, alpha=0.5, label="Our metric")
    ax[i//2, i%2].bar(df_fold["rating"].value_counts().index, df_fold["rating"].value_counts().values, alpha=0.5, label="Rating LLM judge)")
    ax[i//2, i%2].set_title(f"Fold {fold}")
    ax[i//2, i%2].set_xticks(np.arange(0, 2), labels=["0", "1"])
    ax[i//2, i%2].set_xlabel("Rating")
    ax[i//2, i%2].set_ylabel("Count")
    ax[i//2, i%2].legend(["Our metric", "Rating LLM judge"])
plt.tight_layout()
plt.show()